In [ ]:
# 08 — Backfill hours worked and amount paid on the existing incident history (migration 005)
# Adds hours_worked / amount_paid_eur to every completed synthetic incident, in Delta (synth_incidents, all 1M)
# and in Lakebase (the ~45k recent ones loaded by 07). Values come from billing_columns in 00_config: a hash of
# the incident id, so they equal what the generator (05) now produces and re-runs change nothing.
# Lakebase rows that already have billing (entered in the app) are never overwritten.

In [ ]:
%pip install -q "psycopg[binary]>=3.1"

In [ ]:
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [ ]:
%run ./00_config

In [ ]:
# Cell 1 — Delta: add the columns if needed, then MERGE the billing figures into synth_incidents
import time
import psycopg
from pyspark.sql import functions as F

PG_URL      = dbutils.secrets.get("maintops", "lakebase_pg_url")
RECENT_FROM = "2026-06-16"                  # same slice as 07_load_lakebase
started = time.time()

if "hours_worked" not in spark.table(T_INCIDENTS).columns:
    spark.sql(f"ALTER TABLE {T_INCIDENTS} ADD COLUMNS (hours_worked DECIMAL(5,2), amount_paid_eur DECIMAL(10,2))")

billing = billing_columns(spark.table(T_INCIDENTS).select("id", "incident_type", "status", "handyman_user_id", "created_at"),
                          spark.table(T_HANDYMAN_DETAILS))
billing.where("hours_worked IS NOT NULL").createOrReplaceTempView("_billing")
spark.sql(f"""
    MERGE INTO {T_INCIDENTS} t USING _billing s ON t.id = s.id
    WHEN MATCHED AND (t.hours_worked IS DISTINCT FROM s.hours_worked OR t.amount_paid_eur IS DISTINCT FROM s.amount_paid_eur)
      THEN UPDATE SET t.hours_worked = s.hours_worked, t.amount_paid_eur = s.amount_paid_eur""")

inc = spark.table(T_INCIDENTS)
n_completed = inc.where("status = 'completed'").count()
n_billed = inc.where("hours_worked IS NOT NULL").count()
print(f"Delta: {n_billed:,} of {n_completed:,} completed incidents billed ({time.time() - started:.0f}s)")
assert n_billed == n_completed, "every completed incident must have billing"
assert inc.where("status <> 'completed' AND (hours_worked IS NOT NULL OR amount_paid_eur IS NOT NULL)").count() == 0

In [ ]:
# Cell 2 — Lakebase: the same figures for the loaded slice; app-entered billing is kept
recent = (spark.table(T_INCIDENTS)
          .where(F.col("hours_worked").isNotNull() & (F.col("created_at") >= F.lit(RECENT_FROM)))
          .select("id", "hours_worked", "amount_paid_eur"))

with psycopg.connect(PG_URL, connect_timeout=30) as conn, conn.cursor() as cur:
    cur.execute("CREATE TEMP TABLE _billing (id bigint PRIMARY KEY, hours_worked numeric(5,2), "
                "amount_paid_eur numeric(10,2)) ON COMMIT DROP")
    staged = 0
    with cur.copy("COPY _billing (id, hours_worked, amount_paid_eur) FROM STDIN") as cp:
        for row in recent.toLocalIterator():
            cp.write_row(tuple(row))
            staged += 1
    cur.execute("""
        UPDATE maintops.incidents i
        SET hours_worked = b.hours_worked, amount_paid_eur = b.amount_paid_eur
        FROM _billing b
        WHERE i.id = b.id AND i.status = 'completed' AND i.hours_worked IS NULL""")
    updated = cur.rowcount
    cur.execute("""SELECT count(*) FROM _billing b JOIN maintops.incidents i USING (id)
                   WHERE i.status = 'completed' AND i.hours_worked IS NULL""")
    missed = cur.fetchone()[0]
    cur.execute("""SELECT count(*) FROM maintops.incidents
                   WHERE status = 'completed' AND hours_worked IS NULL""")
    unbilled = cur.fetchone()[0]
    assert missed == 0, f"{missed} staged incidents still have no billing"
    conn.commit()

# Completed in the app before migration 005 (not part of the synthetic history): they keep NULL billing
print(f"Lakebase: staged {staged:,}, updated {updated:,}; completed without billing (app, pre-005): {unbilled:,}")